In [ ]:
import json

import pandas as pd
import great_expectations as gx

from agoradatatools.gx import GreatExpectationsRunner

# get_context FIRST, then custom expectation imports
context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange

# Create Expectation Suite for RNA DE Aggregate Data

## Get Example Data File

In [ ]:
# Use the local staging file.
# Alternatively, fetch from Synapse: rna_de_aggregate_file = syn.get("<SYN_ID>").path
rna_de_aggregate_file = "../staging/rna_de_aggregate.json"

## Create Validator Object on Data File

In [ ]:
nested_columns = ["name", "4 months", "12 months", "18 months"]
df = pd.read_json(rna_de_aggregate_file)
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "rna_de_aggregate"

## Add Expectations to Validator Object For Each Column

In [ ]:
# ensembl_gene_id
validator.expect_column_values_to_be_of_type("ensembl_gene_id", "str")
validator.expect_column_values_to_not_be_null("ensembl_gene_id")
validator.expect_column_values_to_match_regex("ensembl_gene_id", r"^ENSMUSG\d{11}$")

In [ ]:
# gene_symbol (can be empty string, never null)
validator.expect_column_values_to_be_of_type("gene_symbol", "str")
validator.expect_column_values_to_not_be_null("gene_symbol")

In [ ]:
# matched_control
validator.expect_column_values_to_be_of_type("matched_control", "str")
validator.expect_column_values_to_not_be_null("matched_control")

In [ ]:
# model_group (nullable — validate type only when not null)
validator.expect_column_values_to_be_of_type(
    "model_group", "str",
    row_condition="model_group.notnull()",
    condition_parser="pandas"
)

In [ ]:
# model_type
validator.expect_column_values_to_be_of_type("model_type", "str")
validator.expect_column_values_to_not_be_null("model_type")
validator.expect_column_values_to_be_in_set("model_type", {"Familial AD", "Late Onset AD"})

In [ ]:
# tissue
validator.expect_column_values_to_be_of_type("tissue", "str")
validator.expect_column_values_to_not_be_null("tissue")
validator.expect_column_values_to_be_in_set("tissue", {"Cerebral Cortex", "Hemibrain", "Hippocampus"})

In [ ]:
# sex_cohort
validator.expect_column_values_to_be_of_type("sex_cohort", "str")
validator.expect_column_values_to_not_be_null("sex_cohort")
validator.expect_column_values_to_be_in_set("sex_cohort", {"Females", "Males", "Females & Males"})

In [ ]:
# biodomains (list of strings, can be empty)
validator.expect_column_values_to_be_of_type("biodomains", "list")
validator.expect_column_values_to_not_be_null("biodomains")
validator.expect_column_values_to_have_list_members_of_type(column="biodomains", member_type="str")
validator.expect_column_values_to_have_list_length_in_range(column="biodomains", list_length_range=[0, 30])

In [ ]:
# name (required link object: {link_url, link_text})
with open("../src/agoradatatools/great_expectations/gx/json_schemas/rna_de_aggregate/name_schema.json") as f:
    name_schema = json.load(f)

validator.expect_column_values_to_be_of_type("name", "str")
validator.expect_column_values_to_not_be_null("name")
validator.expect_column_values_to_match_json_schema("name", json_schema=name_schema)

In [ ]:
# age entry columns (nullable dict: {log2_fc, adj_p_val})
with open("../src/agoradatatools/great_expectations/gx/json_schemas/rna_de_aggregate/age_entry_schema.json") as f:
    age_entry_schema = json.load(f)

validator.expect_column_values_to_be_of_type("4 months", "str")
validator.expect_column_values_to_match_json_schema("4 months", json_schema=age_entry_schema)

validator.expect_column_values_to_be_of_type("12 months", "str")
validator.expect_column_values_to_match_json_schema("12 months", json_schema=age_entry_schema)

validator.expect_column_values_to_be_of_type("18 months", "str")
validator.expect_column_values_to_match_json_schema("18 months", json_schema=age_entry_schema)

In [ ]:
# each (gene, model, tissue, sex) combination should appear exactly once
validator.expect_compound_columns_to_be_unique(["ensembl_gene_id", "name", "tissue", "sex_cohort"])

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()